# ⚗️ Notebook 02 — Data Preprocessing & Feature Engineering
## MarocChurn · Telecom Customer Churn Prediction
**Author:** Ahmed Mansof · Master Data Science · ENS Tétouan

---

### 🎯 Objectives
1. Handle missing values
2. Encode categorical variables
3. Create new engineered features
4. Normalize numerical features
5. Handle class imbalance with SMOTE
6. Save the clean dataset

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

plt.rcParams.update({
    'figure.facecolor': '#0d1e2d', 'axes.facecolor': '#0d1e2d',
    'axes.edgecolor': '#1a3348', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white', 'text.color': 'white',
})
CYAN = '#00e5ff'; PURPLE = '#7b61ff'; ORANGE = '#ff6b35'

print("✅ Imports OK")

## 1. Load Raw Data

In [ ]:
import os, sys
sys.path.insert(0, '..')

DATA_PATH = '../data/raw/telco_churn.csv'
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    df.columns = df.columns.str.strip()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    print(f"✅ Real data: {df.shape}")
else:
    np.random.seed(42); n = 7043
    tenures  = np.random.randint(0, 73, n)
    monthly  = np.round(np.random.uniform(18, 119, n), 2)
    contracts= np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.24,0.21])
    churn_p  = np.clip(0.05+0.35*(contracts=='Month-to-month')+0.20*(tenures<6)+0.15*(monthly>80),0,1)
    churn    = np.array(['Yes' if np.random.random()<p else 'No' for p in churn_p])
    df = pd.DataFrame({
        'customerID': [f'C{i:05d}' for i in range(n)], 'gender': np.random.choice(['Male','Female'],n),
        'SeniorCitizen': np.random.choice([0,1],n,p=[0.84,0.16]), 'Partner': np.random.choice(['Yes','No'],n),
        'Dependents': np.random.choice(['Yes','No'],n,p=[0.3,0.7]), 'tenure': tenures,
        'PhoneService': np.random.choice(['Yes','No'],n,p=[0.9,0.1]),
        'MultipleLines': np.random.choice(['Yes','No','No phone service'],n),
        'InternetService': np.random.choice(['Fiber optic','DSL','No'],n,p=[0.44,0.34,0.22]),
        'OnlineSecurity': np.random.choice(['Yes','No','No internet service'],n),
        'OnlineBackup': np.random.choice(['Yes','No','No internet service'],n),
        'DeviceProtection': np.random.choice(['Yes','No','No internet service'],n),
        'TechSupport': np.random.choice(['Yes','No','No internet service'],n),
        'StreamingTV': np.random.choice(['Yes','No','No internet service'],n),
        'StreamingMovies': np.random.choice(['Yes','No','No internet service'],n),
        'Contract': contracts, 'PaperlessBilling': np.random.choice(['Yes','No'],n,p=[0.59,0.41]),
        'PaymentMethod': np.random.choice(['Electronic check','Mailed check','Bank transfer (automatic)','Credit card (automatic)'],n),
        'MonthlyCharges': monthly,
        'TotalCharges': np.round(tenures*monthly+np.random.uniform(-50,50,n),2), 'Churn': churn
    })
    print(f"⚠️  Synthetic data: {df.shape}")

print(df.dtypes)

## 2. Handle Missing Values

In [ ]:
# ── Step 1 — Missing values ──────────────────────────────────
print("Missing values BEFORE:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill TotalCharges with median (11 rows = customers with 0 tenure)
median_total = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_total, inplace=True)

# Drop customerID (not predictive)
df.drop(columns=['customerID'], inplace=True, errors='ignore')

print("\nMissing values AFTER:")
print(df.isnull().sum().sum(), "missing values remaining")
print("✅ Missing values handled")

## 3. Feature Engineering — Create New Features

In [ ]:
# ── Step 2 — Feature Engineering ────────────────────────────
print("Features BEFORE engineering:", df.shape[1], "columns")

# Feature 1: charges_ratio — how much does the customer pay relative to tenure?
# → High ratio = paying a lot from the start → risk signal
df['charges_ratio'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# Feature 2: is_new_customer — first 6 months are the riskiest period
df['is_new_customer'] = (df['tenure'] < 6).astype(int)

# Feature 3: high_value — high monthly charges customers who churn are more costly
df['high_value'] = (df['MonthlyCharges'] > 70).astype(int)

# Feature 4: has_support_services — customers with support tend to stay
df['has_online_security'] = (df['OnlineSecurity'] == 'Yes').astype(int)
df['has_tech_support']    = (df['TechSupport'] == 'Yes').astype(int)

print("Features AFTER engineering:", df.shape[1], "columns")
print("\nNew features created:")
print("  ✅ charges_ratio   = MonthlyCharges / (tenure + 1)")
print("  ✅ is_new_customer = 1 if tenure < 6 months")
print("  ✅ high_value      = 1 if MonthlyCharges > 70$")
print("  ✅ has_online_security")
print("  ✅ has_tech_support")

# Visualize new features vs churn
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Engineered Features vs Churn', fontsize=13)

target = (df['Churn'] == 'Yes') if df['Churn'].dtype == 'object' else df['Churn']

for ax, feat, title in zip(
    axes,
    ['charges_ratio', 'is_new_customer', 'high_value'],
    ['Charges Ratio', 'Is New Customer (0/1)', 'High Value (0/1)']
):
    if df[feat].nunique() > 5:
        ax.hist(df[feat][target==0], bins=30, alpha=0.7, color=CYAN, label='No Churn', density=True)
        ax.hist(df[feat][target==1], bins=30, alpha=0.7, color=ORANGE, label='Churn', density=True)
    else:
        ct = df.groupby(feat)['Churn'].apply(lambda x: (x=='Yes').mean()*100) if df['Churn'].dtype=='object' else df.groupby(feat)['Churn'].mean()*100
        ax.bar(ct.index.astype(str), ct.values, color=[CYAN, ORANGE][:len(ct)])
        ax.set_ylabel('Churn Rate (%)')
    ax.set_title(title)
    ax.legend(facecolor='#0d1e2d') if df[feat].nunique()>5 else None

plt.tight_layout()
plt.savefig('../data/processed/plot_engineered_features.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

## 4. Encode Categorical Variables

In [ ]:
# ── Step 3 — Encode categorical variables ────────────────────
# Convert target
df['Churn'] = (df['Churn'].astype(str).str.strip() == 'Yes').astype(int)

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"Churn rate: {y.mean()*100:.1f}%")

# Encode all categorical columns with LabelEncoder
cat_cols = X.select_dtypes(include='object').columns.tolist()
print(f"\nCategorical columns to encode ({len(cat_cols)}): {cat_cols}")

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

print("\n✅ Encoding done — all columns are now numeric")
print(X.dtypes.value_counts())

## 5. Normalize Numerical Features

In [ ]:
# ── Step 4 — Scale numerical features ────────────────────────
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_ratio']

print("BEFORE scaling:")
print(X[num_cols].describe().round(2))

scaler = MinMaxScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print("\nAFTER scaling (MinMaxScaler → range [0, 1]):")
print(X[num_cols].describe().round(4))

# Save scaler for use in app.py
os.makedirs('../models', exist_ok=True)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("\n✅ Scaler saved to models/scaler.pkl")

## 6. Train/Test Split + SMOTE

In [ ]:
# ── Step 5 — Split & handle imbalance ────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("BEFORE SMOTE:")
print(f"  Train: {X_train.shape} | Churn: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"  Test : {X_test.shape}  | Churn: {y_test.sum()}  ({y_test.mean()*100:.1f}%)")

# Apply SMOTE only on training data (NEVER on test!)
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("\nAFTER SMOTE (training only):")
print(f"  Train: {X_train_res.shape} | Churn: {y_train_res.sum()} ({y_train_res.mean()*100:.1f}%)")

# Visualize class balance
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (labels, values), title in zip(
    axes,
    [(y_train.value_counts().index, y_train.value_counts().values),
     (y_train_res.value_counts().index, y_train_res.value_counts().values)],
    ['Before SMOTE', 'After SMOTE']
):
    ax.bar(['No Churn', 'Churn'], values, color=[CYAN, ORANGE], edgecolor='none', width=0.5)
    for i, v in enumerate(values):
        ax.text(i, v + 50, str(v), ha='center', color='white', fontweight='bold')
    ax.set_title(title); ax.set_ylabel('Count')
    ax.set_ylim(0, max(values)*1.15)
plt.suptitle('Class Balance: Before vs After SMOTE', fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/plot_smote_balance.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

## 7. Save Processed Data

In [ ]:
# ── Step 6 — Save everything ─────────────────────────────────
os.makedirs('../data/processed', exist_ok=True)

# Save clean dataset
df_clean = X.copy()
df_clean['Churn'] = y
df_clean.to_csv('../data/processed/telco_clean.csv', index=False)

# Save train/test splits
X_train_res.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
pd.Series(y_train_res).to_csv('../data/processed/y_train.csv', index=False)
pd.Series(y_test).to_csv('../data/processed/y_test.csv', index=False)

# Save feature names
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print("✅ Files saved:")
print("   data/processed/telco_clean.csv")
print("   data/processed/X_train.csv + y_train.csv")
print("   data/processed/X_test.csv  + y_test.csv")
print("   models/scaler.pkl")
print("   models/feature_names.pkl")
print(f"\nFinal dataset: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"Feature count: {X.shape[1]}")
print("\n✅ Preprocessing complete. Proceed to 03_modeling.ipynb")